# Phase 3 Part 1: LightGCN on All 5 Datasets
**CSC14114 - Big Data Applications | Official LightGCN GitHub repo workflow**

This notebook is **Phase 3 - Part 1** only.

## Phase 3 split
1. Part 1: run **LightGCN** on all 5 datasets
2. Part 2: run **SGL** on all 5 datasets
3. Part 3: run **SimGCL** on all 5 datasets and compare all 3 models

## What this notebook does
- uses the official PyTorch LightGCN repository: `gusye1234/LightGCN-PyTorch`
- downloads benchmark datasets from official / cited sources
- uses your **Phase 2 Last.fm dataset** as the 5th dataset
- runs **3 repeated runs with fixed seed 42 per dataset** and reports mean/std
- computes `Recall@20`, `NDCG@20`, `HR@20`, `MRR@20`, supports early stopping, and saves a 3-run average report folder
- saves logs, metrics, runtime, and convergence curves

## Datasets used in Part 1
- `ml-1m`
- `gowalla`
- `yelp2018`
- `amazon-book`
- `lastfm_phase2`

## Kaggle setup
- Turn **Internet = ON**
- Turn **GPU = ON**
- Upload this notebook
- Add a Kaggle input dataset that contains either:
  - `lastfm_phase2_processed.zip`, or
  - extracted files `interactions.csv`, `users.csv`, `items.csv`, `manifest.json`
- Then click **Run All** and **Save Version**


In [ ]:
import ast
import base64
import io
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch

ROOT = Path('/kaggle/working/phase3_part1_lightgcn') if Path('/kaggle').exists() else Path('phase3_part1_lightgcn')
WORK = ROOT / 'workspace'
REPO_DIR = WORK / 'LightGCN-PyTorch'
RESULTS_DIR = ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
AVG_RESULTS_DIR = RESULTS_DIR / 'average_results_3_runs'
INPUT_DIR = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
for p in [WORK, RESULTS_DIR, FIG_DIR, AVG_RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Workspace:', ROOT.resolve())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
LASTFM_REQUIRED_FILES = ['interactions.csv', 'users.csv', 'items.csv', 'manifest.json']


def find_lastfm_phase2_source(input_dir: Path):
    zip_candidates = sorted(input_dir.rglob('*.zip'))
    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path) as zf:
                names = {Path(name).name for name in zf.namelist() if not name.endswith('/')}
            if all(name in names for name in LASTFM_REQUIRED_FILES):
                return ('zip', zip_path)
        except zipfile.BadZipFile:
            continue

    seen = set()
    for manifest_path in sorted(input_dir.rglob('manifest.json')):
        folder = manifest_path.parent
        folder_key = str(folder.resolve())
        if folder_key in seen:
            continue
        seen.add(folder_key)
        if all((folder / name).exists() for name in LASTFM_REQUIRED_FILES):
            return ('dir', folder)

    raise FileNotFoundError(
        'Could not find Last.fm Phase 2 input data under /kaggle/input. '
        'Attach a Kaggle dataset containing lastfm_phase2_processed.zip or the extracted processed CSV/JSON files.'
    )


def materialize_lastfm_phase2_input() -> Path:
    target_dir = WORK / 'lastfm_phase2_input'
    if target_dir.exists() and all((target_dir / name).exists() for name in LASTFM_REQUIRED_FILES):
        return target_dir

    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    source_kind, source_path = find_lastfm_phase2_source(INPUT_DIR)
    if source_kind == 'zip':
        with zipfile.ZipFile(source_path) as zf:
            zf.extractall(target_dir)
    else:
        for name in LASTFM_REQUIRED_FILES:
            shutil.copy2(source_path / name, target_dir / name)

    return target_dir


LASTFM_PHASE2_INPUT_DIR = materialize_lastfm_phase2_input()
print(f'Last.fm Phase 2 input ready at: {LASTFM_PHASE2_INPUT_DIR}')


In [ ]:
REPO_URL = 'https://github.com/gusye1234/LightGCN-PyTorch.git'
DATASETS = ['ml-1m', 'gowalla', 'yelp2018', 'amazon-book', 'lastfm_phase2']
FIXED_SEED = 42
RUN_IDS = [1, 2, 3]
TOPKS = [10, 20]
REPORT_TOPK = 20
EARLY_STOPPING = {
    'enabled': True,
    'monitor_metric': 'recall',
    'monitor_topk': 20,
    'patience_evals': 5,
    'min_delta': 0.0,
}

DATASET_RUN_CONFIG = {
    'ml-1m': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'gowalla': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'yelp2018': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'amazon-book': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'lastfm_phase2': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
}

RAW_SPLIT_URLS = {
    'gowalla': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/test.txt',
    },
    'yelp2018': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/test.txt',
    },
    'amazon-book': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/test.txt',
    },
}

MOVIELENS_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'


In [ ]:
def run_cmd(cmd, cwd=None, capture_output=False):
    print('$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=capture_output)


def download_file(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'[skip] {dest.name}')
        return dest
    print(f'[download] {url}')
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with dest.open('wb') as f:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
    return dest


def write_repo_split(folder: Path, train_map: dict, test_map: dict):
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / 'train.txt').open('w', encoding='utf-8') as f:
        for uid in sorted(train_map):
            items = train_map[uid]
            if items:
                f.write(str(uid) + ' ' + ' '.join(str(i) for i in items) + '\n')
    with (folder / 'test.txt').open('w', encoding='utf-8') as f:
        for uid in sorted(test_map):
            items = test_map[uid]
            if items:
                f.write(str(uid) + ' ' + ' '.join(str(i) for i in items) + '\n')


def build_leave_one_out_maps(df: pd.DataFrame, user_col: str, item_col: str, time_col: str):
    work = df[[user_col, item_col, time_col]].copy()
    work = work.sort_values([user_col, time_col, item_col], kind='mergesort')
    users = sorted(work[user_col].unique().tolist())
    items = sorted(work[item_col].unique().tolist())
    u_map = {u: i for i, u in enumerate(users)}
    i_map = {it: i for i, it in enumerate(items)}
    work['uid'] = work[user_col].map(u_map).astype(int)
    work['iid'] = work[item_col].map(i_map).astype(int)

    train_map = {}
    test_map = {}
    for uid, group in work.groupby('uid', sort=True):
        item_list = group['iid'].tolist()
        if len(item_list) < 2:
            continue
        train_map[int(uid)] = [int(x) for x in item_list[:-1]]
        test_map[int(uid)] = [int(item_list[-1])]
    return train_map, test_map


def clone_and_patch_repo():
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', REPO_URL, str(REPO_DIR)])
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'tensorboardX', 'cppimport', 'pybind11'])

    world_path = REPO_DIR / 'code' / 'world.py'
    register_path = REPO_DIR / 'code' / 'register.py'

    world_text = world_path.read_text(encoding='utf-8')
    world_text = world_text.replace(
        "all_dataset = ['lastfm', 'gowalla', 'yelp2018', 'amazon-book']",
        "all_dataset = ['lastfm', 'gowalla', 'yelp2018', 'amazon-book', 'ml-1m', 'lastfm_phase2']"
    )
    world_text = world_text.replace(
        "FILE_PATH = join(CODE_PATH, 'checkpoints')",
        "FILE_PATH = args.path"
    )
    world_path.write_text(world_text, encoding='utf-8')

    register_text = '\n'.join([
        'import world',
        'import dataloader',
        'import model',
        'import utils',
        'from pprint import pprint',
        '',
        "if world.dataset == 'lastfm':",
        '    dataset = dataloader.LastFM()',
        'else:',
        "    dataset = dataloader.Loader(path='../data/' + world.dataset)",
        '',
        "print('===========config================')",
        'pprint(world.config)',
        "print('cores for test:', world.CORES)",
        "print('comment:', world.comment)",
        "print('tensorboard:', world.tensorboard)",
        "print('LOAD:', world.LOAD)",
        "print('Weight path:', world.PATH)",
        "print('Test Topks:', world.topks)",
        "print('using bpr loss')",
        "print('===========end===================')",
        'MODELS = {',
        "    'mf': model.PureMF,",
        "    'lgn': model.LightGCN,",
        '}',
        '',
    ])
    register_path.write_text(register_text, encoding='utf-8')

    metric_eval_script = '''
import argparse
import ast
import json
import random
import sys

import numpy as np
import torch

def build_cli():
    parser = argparse.ArgumentParser(description='Evaluate a saved LightGCN checkpoint with HR/MRR metrics.')
    parser.add_argument('--dataset', required=True)
    parser.add_argument('--checkpoint', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--topks', required=True)
    parser.add_argument('--testbatch', type=int, default=100)
    parser.add_argument('--seed', type=int, default=2020)
    parser.add_argument('--bpr_batch', type=int, default=2048)
    parser.add_argument('--recdim', type=int, default=64)
    parser.add_argument('--layer', type=int, default=3)
    parser.add_argument('--lr', type=float, default=0.001)
    parser.add_argument('--decay', type=float, default=1e-4)
    parser.add_argument('--model', default='lgn')
    return parser.parse_args()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

def batch_iter(values, batch_size):
    for idx in range(0, len(values), batch_size):
        yield values[idx: idx + batch_size]

def get_label(test_data, pred_data):
    rows = []
    for idx, ground_truth in enumerate(test_data):
        rows.append(np.array([float(item in ground_truth) for item in pred_data[idx]], dtype=float))
    return np.array(rows, dtype=float)

def hr_at_k(r, k):
    pred_data = r[:, :k]
    hits = (pred_data.sum(axis=1) > 0).astype(float)
    return float(np.sum(hits))

def mrr_at_k(r, k):
    pred_data = r[:, :k]
    hit_mask = pred_data > 0
    first_hit = np.where(hit_mask.any(axis=1), hit_mask.argmax(axis=1) + 1, 0)
    reciprocal = np.where(first_hit > 0, 1.0 / first_hit, 0.0)
    return float(np.sum(reciprocal))

args = build_cli()
topks = list(ast.literal_eval(args.topks))
sys.argv = [
    'metric_eval.py',
    f'--dataset={args.dataset}',
    f'--seed={args.seed}',
    f'--bpr_batch={args.bpr_batch}',
    f'--recdim={args.recdim}',
    f'--layer={args.layer}',
    f'--lr={args.lr}',
    f'--decay={args.decay}',
    f'--testbatch={args.testbatch}',
    '--tensorboard=0',
    '--multicore=0',
    f'--model={args.model}',
    f'--topks={repr(topks)}',
    '--comment=metric-eval',
    '--epochs=1',
]

import world
import register
from register import dataset

set_seed(args.seed)
recmodel = register.MODELS[world.model_name](world.config, dataset)
state_dict = torch.load(args.checkpoint, map_location=torch.device('cpu'))
recmodel.load_state_dict(state_dict)
recmodel = recmodel.to(world.device)
recmodel.eval()

users = list(dataset.testDict.keys())
max_k = max(topks)
ground_truth_list = []
rating_list = []

with torch.no_grad():
    for batch_users in batch_iter(users, world.config['test_u_batch_size']):
        all_pos = dataset.getUserPosItems(batch_users)
        ground_truth = [dataset.testDict[int(u)] for u in batch_users]
        batch_users_gpu = torch.tensor(batch_users, dtype=torch.long, device=world.device)
        rating = recmodel.getUsersRating(batch_users_gpu)
        exclude_index = []
        exclude_items = []
        for row_idx, items in enumerate(all_pos):
            exclude_index.extend([row_idx] * len(items))
            exclude_items.extend(items)
        if exclude_index:
            rating[exclude_index, exclude_items] = -(1 << 10)
        _, rating_k = torch.topk(rating, k=max_k)
        rating_list.append(rating_k.cpu())
        ground_truth_list.extend(ground_truth)

pred_data = torch.cat(rating_list, dim=0).numpy() if rating_list else np.empty((0, max_k), dtype=int)
r = get_label(ground_truth_list, pred_data) if len(ground_truth_list) else np.empty((0, max_k), dtype=float)
user_count = max(len(users), 1)

metrics = {}
for k in topks:
    metrics[f'hr@{k}'] = hr_at_k(r, k) / user_count if len(r) else 0.0
    metrics[f'mrr@{k}'] = mrr_at_k(r, k) / user_count if len(r) else 0.0

with open(args.output, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))
'''.lstrip()
    metric_eval_path = REPO_DIR / 'code' / 'metric_eval.py'
    metric_eval_path.write_text(metric_eval_script, encoding='utf-8')

    main_earlystop_script = '''
import json
import os
import shutil
import time
from pathlib import Path
from os.path import join

import torch
from tensorboardX import SummaryWriter

import Procedure
import utils
import world
from world import cprint

def metrics_to_map(results):
    metric_map = {}
    for idx, k in enumerate(world.topks):
        metric_map[f'precision@{k}'] = float(results['precision'][idx])
        metric_map[f'recall@{k}'] = float(results['recall'][idx])
        metric_map[f'ndcg@{k}'] = float(results['ndcg'][idx])
    return metric_map

utils.set_seed(world.seed)
print('>>SEED:', world.seed)

import register
from register import dataset

early_stop_enabled = os.getenv('LIGHTGCN_EARLY_STOP_ENABLED', '1') == '1'
monitor_metric = os.getenv('LIGHTGCN_EARLY_STOP_METRIC', 'recall').lower()
monitor_topk = int(os.getenv('LIGHTGCN_EARLY_STOP_TOPK', '20'))
patience_evals = int(os.getenv('LIGHTGCN_EARLY_STOP_PATIENCE', '5'))
min_delta = float(os.getenv('LIGHTGCN_EARLY_STOP_MIN_DELTA', '0.0'))

if monitor_metric not in {'precision', 'recall', 'ndcg'}:
    raise ValueError(f'Unsupported early-stop metric: {monitor_metric}')
if monitor_topk not in world.topks:
    raise ValueError(f'early-stop topk={monitor_topk} must be in world.topks={world.topks}')

monitor_key = f'{monitor_metric}@{monitor_topk}'
run_dir = Path(world.FILE_PATH).resolve().parent
best_checkpoint_path = Path(world.FILE_PATH).resolve() / 'best_model.pth.tar'
early_stop_summary_path = run_dir / 'early_stop_summary.json'

Recmodel = register.MODELS[world.model_name](world.config, dataset)
Recmodel = Recmodel.to(world.device)
bpr = utils.BPRLoss(Recmodel, world.config)
weight_file = utils.getFileName()
print(f'load and save to {weight_file}')
if world.LOAD:
    try:
        Recmodel.load_state_dict(torch.load(weight_file, map_location=torch.device('cpu')))
        world.cprint(f'loaded model weights from {weight_file}')
    except FileNotFoundError:
        print(f'{weight_file} not exists, start from beginning')

Neg_k = 1
if world.tensorboard:
    w = SummaryWriter(join(world.BOARD_PATH, time.strftime('%m-%d-%Hh%Mm%Ss-') + '-' + world.comment))
else:
    w = None
world.cprint('not enable tensorflowboard')

best_score = None
best_epoch = None
best_metrics = None
bad_eval_count = 0
stop_epoch = None
stopped_early = False

try:
    for epoch in range(world.TRAIN_epochs):
        if epoch % 10 == 0:
            cprint('[TEST]')
            test_results = Procedure.Test(dataset, Recmodel, epoch, w, world.config['multicore'])
            metric_map = metrics_to_map(test_results)
            current_score = metric_map[monitor_key]
            improved = best_score is None or current_score > (best_score + min_delta)
            if improved:
                best_score = current_score
                best_epoch = epoch
                best_metrics = metric_map
                bad_eval_count = 0
                torch.save(Recmodel.state_dict(), weight_file)
                torch.save(Recmodel.state_dict(), best_checkpoint_path)
                print(f'[EARLY_STOP] new best {monitor_key}={current_score:.6f} at eval epoch {epoch}')
            else:
                bad_eval_count += 1
                print(
                    f'[EARLY_STOP] no improvement on {monitor_key}: '
                    f'{current_score:.6f} vs best {best_score:.6f} '
                    f'(bad_eval_count={bad_eval_count}/{patience_evals})'
                )
                if early_stop_enabled and bad_eval_count >= patience_evals:
                    stopped_early = True
                    stop_epoch = epoch
                    print(f'[EARLY_STOP] stopping at eval epoch {epoch}')
                    break
        output_information = Procedure.BPR_train_original(dataset, Recmodel, bpr, epoch, neg_k=Neg_k, w=w)
        print(f'EPOCH[{epoch+1}/{world.TRAIN_epochs}] {output_information}')
        torch.save(Recmodel.state_dict(), weight_file)
    if best_checkpoint_path.exists():
        shutil.copy2(best_checkpoint_path, weight_file)
finally:
    if world.tensorboard:
        w.close()
    summary = {
        'early_stop_enabled': early_stop_enabled,
        'monitor_metric': monitor_metric,
        'monitor_topk': monitor_topk,
        'monitor_key': monitor_key,
        'patience_evals': patience_evals,
        'min_delta': min_delta,
        'stopped_early': stopped_early,
        'stop_epoch': stop_epoch,
        'best_epoch': best_epoch,
        'best_score': best_score,
        'best_metrics': best_metrics,
        'best_checkpoint_path': str(best_checkpoint_path),
        'checkpoint_path': weight_file,
    }
    early_stop_summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
'''.lstrip()
    main_earlystop_path = REPO_DIR / 'code' / 'main_earlystop.py'
    main_earlystop_path.write_text(main_earlystop_script, encoding='utf-8')
    print('Official LightGCN repo is ready with extra metric evaluator and early stopping entrypoint.')


In [ ]:
def prepare_standard_repo_splits():
    data_root = REPO_DIR / 'data'
    for name, urls in RAW_SPLIT_URLS.items():
        folder = data_root / name
        folder.mkdir(parents=True, exist_ok=True)
        download_file(urls['train'], folder / 'train.txt')
        download_file(urls['test'], folder / 'test.txt')


def prepare_ml_1m_split():
    folder = REPO_DIR / 'data' / 'ml-1m'
    if (folder / 'train.txt').exists() and (folder / 'test.txt').exists():
        print('ml-1m already prepared')
        return
    zip_path = download_file(MOVIELENS_URL, WORK / 'ml-1m.zip')
    extract_dir = WORK / 'ml-1m_extracted'
    if not extract_dir.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
    ratings_path = extract_dir / 'ml-1m' / 'ratings.dat'
    df = pd.read_csv(
        ratings_path,
        sep='::',
        header=None,
        names=['user_id', 'item_id', 'rating', 'timestamp'],
        engine='python',
        encoding='latin-1',
    )
    df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
    train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
    write_repo_split(folder, train_map, test_map)
    print('Prepared ml-1m')


def prepare_lastfm_phase2_split():
    folder = REPO_DIR / 'data' / 'lastfm_phase2'
    if (folder / 'train.txt').exists() and (folder / 'test.txt').exists():
        print('lastfm_phase2 already prepared')
        return
    df = pd.read_csv(LASTFM_PHASE2_INPUT_DIR / 'interactions.csv')
    df['timestamp'] = pd.to_datetime(df['crawl_timestamp_utc'], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp']).copy()
    df['timestamp'] = df['timestamp'].astype('int64') // 10**9
    df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
    train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
    write_repo_split(folder, train_map, test_map)
    print('Prepared lastfm_phase2')


clone_and_patch_repo()
prepare_standard_repo_splits()
prepare_ml_1m_split()
prepare_lastfm_phase2_split()

print('Prepared datasets:')
for p in sorted((REPO_DIR / 'data').iterdir()):
    if p.is_dir():
        print('-', p.name)


In [ ]:
def parse_metric_history(log_text: str):
    pattern = re.compile(r"\{'precision': array\(\[(.*?)\]\), 'recall': array\(\[(.*?)\]\), 'ndcg': array\(\[(.*?)\]\)\}", re.S)
    rows = []
    for idx, match in enumerate(pattern.findall(log_text)):
        prec = [float(x.strip()) for x in match[0].split(',') if x.strip()]
        rec = [float(x.strip()) for x in match[1].split(',') if x.strip()]
        ndcg = [float(x.strip()) for x in match[2].split(',') if x.strip()]
        epoch = idx * 10
        row = {'eval_epoch': epoch}
        for i, k in enumerate(TOPKS):
            row[f'precision@{k}'] = prec[i]
            row[f'recall@{k}'] = rec[i]
            row[f'ndcg@{k}'] = ndcg[i]
        rows.append(row)
    return rows


def find_checkpoint(ckpt_dir: Path) -> Path:
    best_path = ckpt_dir / 'best_model.pth.tar'
    if best_path.exists():
        return best_path
    checkpoints = sorted(ckpt_dir.glob('*.pth.tar'))
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint found in {ckpt_dir}')
    return checkpoints[-1]


def evaluate_extra_metrics(dataset: str, run_id: int, cfg: dict, run_dir: Path, ckpt_dir: Path):
    checkpoint_path = find_checkpoint(ckpt_dir).resolve()
    metrics_path = (run_dir / 'ranking_metrics.json').resolve()
    cmd = [
        sys.executable, 'metric_eval.py',
        f'--dataset={dataset}',
        f'--checkpoint={checkpoint_path.as_posix()}',
        f'--output={metrics_path.as_posix()}',
        f'--topks={json.dumps(TOPKS)}',
        f"--testbatch={cfg['testbatch']}",
        f'--seed={FIXED_SEED}',
        f"--bpr_batch={cfg['bpr_batch']}",
        f"--recdim={cfg['recdim']}",
        f"--layer={cfg['layer']}",
        f"--lr={cfg['lr']}",
        f"--decay={cfg['decay']}",
        '--model=lgn',
    ]
    proc = subprocess.run(cmd, cwd=REPO_DIR / 'code', text=True, capture_output=True)
    eval_log_path = run_dir / 'metric_eval.log'
    eval_log_path.write_text(proc.stdout + '\n' + proc.stderr, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'Extra metric evaluation failed for dataset={dataset}, run_{run_id}, seed={FIXED_SEED}. Check {eval_log_path}.')
    return json.loads(metrics_path.read_text(encoding='utf-8'))


def run_one(dataset: str, run_id: int):
    cfg = DATASET_RUN_CONFIG[dataset]
    run_name = f'run_{run_id}'
    run_dir = RESULTS_DIR / dataset / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = run_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, 'main_earlystop.py',
        f'--dataset={dataset}',
        f'--seed={FIXED_SEED}',
        f"--epochs={cfg['epochs']}",
        f"--bpr_batch={cfg['bpr_batch']}",
        f"--recdim={cfg['recdim']}",
        f"--layer={cfg['layer']}",
        f"--lr={cfg['lr']}",
        f"--decay={cfg['decay']}",
        f"--testbatch={cfg['testbatch']}",
        '--tensorboard=0',
        '--multicore=0',
        '--model=lgn',
        f'--topks={json.dumps(TOPKS)}',
        f'--path={ckpt_dir.resolve().as_posix()}',
        f'--comment={dataset}-{run_name}-seed-{FIXED_SEED}',
    ]
    print('=' * 90)
    print(f'Running official LightGCN repo: dataset={dataset}, {run_name}, seed={FIXED_SEED}')
    print('=' * 90)
    start = time.time()
    env = os.environ.copy()
    env.update({
        'LIGHTGCN_EARLY_STOP_ENABLED': '1' if EARLY_STOPPING['enabled'] else '0',
        'LIGHTGCN_EARLY_STOP_METRIC': str(EARLY_STOPPING['monitor_metric']),
        'LIGHTGCN_EARLY_STOP_TOPK': str(EARLY_STOPPING['monitor_topk']),
        'LIGHTGCN_EARLY_STOP_PATIENCE': str(EARLY_STOPPING['patience_evals']),
        'LIGHTGCN_EARLY_STOP_MIN_DELTA': str(EARLY_STOPPING['min_delta']),
    })
    proc = subprocess.run(cmd, cwd=REPO_DIR / 'code', text=True, capture_output=True, env=env)
    elapsed = round(time.time() - start, 2)
    log_text = proc.stdout + '\n' + proc.stderr
    log_path = run_dir / 'train.log'
    log_path.write_text(log_text, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'LightGCN failed for dataset={dataset}, {run_name}, seed={FIXED_SEED}. Check {log_path}.')
    history = parse_metric_history(log_text)
    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / 'history.csv', index=False)
    early_stop_summary = {}
    early_stop_summary_path = run_dir / 'early_stop_summary.json'
    if early_stop_summary_path.exists():
        early_stop_summary = json.loads(early_stop_summary_path.read_text(encoding='utf-8'))
    final = early_stop_summary.get('best_metrics') or (history[-1] if history else {})
    extra_metrics = evaluate_extra_metrics(dataset, run_id, cfg, run_dir, ckpt_dir)
    result = {
        'dataset': dataset,
        'model': 'LightGCN',
        'run_id': run_id,
        'run_name': run_name,
        'seed': FIXED_SEED,
        'stopped_early': bool(early_stop_summary.get('stopped_early', False)),
        'best_eval_epoch': early_stop_summary.get('best_epoch'),
        'stop_eval_epoch': early_stop_summary.get('stop_epoch'),
        'early_stop_monitor': early_stop_summary.get('monitor_key'),
        'elapsed_seconds': elapsed,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        **final,
        **extra_metrics,
    }
    (run_dir / 'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result


## Run all Phase 3 Part 1 experiments
This runs **5 datasets x 3 repeated runs with fixed seed 42 = 15 runs** using the official LightGCN repo.


In [ ]:
all_results = []
for dataset in DATASETS:
    for run_id in RUN_IDS:
        all_results.append(run_one(dataset, run_id))

results_df = pd.DataFrame(all_results).sort_values(['dataset', 'run_id']).reset_index(drop=True)
results_df.to_csv(RESULTS_DIR / 'all_runs.csv', index=False)
display(results_df)


In [ ]:
metric_cols = [c for c in results_df.columns if '@' in c] + ['elapsed_seconds']
summary_rows = []
for dataset, group in results_df.groupby('dataset', sort=True):
    row = {'dataset': dataset, 'model': 'LightGCN', 'runs': int(len(group))}
    for col in metric_cols:
        row[f'{col}_mean'] = float(group[col].mean())
        row[f'{col}_std'] = float(group[col].std(ddof=0))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('dataset').reset_index(drop=True)
summary_df.to_csv(RESULTS_DIR / 'summary.csv', index=False)
summary_df.to_csv(AVG_RESULTS_DIR / 'summary_with_std.csv', index=False)

incomplete = summary_df.loc[summary_df['runs'] != len(RUN_IDS), 'dataset'].tolist()
if incomplete:
    raise RuntimeError(f'Cannot build 3-run averages. Incomplete datasets: {incomplete}')

average_report_df = pd.DataFrame({
    'Dataset': summary_df['dataset'],
    'Model': summary_df['model'],
    f'Recall@{REPORT_TOPK}': summary_df[f'recall@{REPORT_TOPK}_mean'].round(6),
    f'NDCG@{REPORT_TOPK}': summary_df[f'ndcg@{REPORT_TOPK}_mean'].round(6),
    f'HR@{REPORT_TOPK}': summary_df[f'hr@{REPORT_TOPK}_mean'].round(6),
    f'MRR@{REPORT_TOPK}': summary_df[f'mrr@{REPORT_TOPK}_mean'].round(6),
})
average_report_df.to_csv(AVG_RESULTS_DIR / 'average_metrics_3_runs.csv', index=False)
(AVG_RESULTS_DIR / 'average_metrics_3_runs.json').write_text(
    json.dumps(average_report_df.to_dict(orient='records'), indent=2),
    encoding='utf-8'
)

for row in summary_df.to_dict(orient='records'):
    dataset_payload = {
        'dataset': row['dataset'],
        'model': row['model'],
        'runs': row['runs'],
        'mean': {
            f'recall@{REPORT_TOPK}': row[f'recall@{REPORT_TOPK}_mean'],
            f'ndcg@{REPORT_TOPK}': row[f'ndcg@{REPORT_TOPK}_mean'],
            f'hr@{REPORT_TOPK}': row[f'hr@{REPORT_TOPK}_mean'],
            f'mrr@{REPORT_TOPK}': row[f'mrr@{REPORT_TOPK}_mean'],
            'elapsed_seconds': row['elapsed_seconds_mean'],
        },
        'std': {
            f'recall@{REPORT_TOPK}': row[f'recall@{REPORT_TOPK}_std'],
            f'ndcg@{REPORT_TOPK}': row[f'ndcg@{REPORT_TOPK}_std'],
            f'hr@{REPORT_TOPK}': row[f'hr@{REPORT_TOPK}_std'],
            f'mrr@{REPORT_TOPK}': row[f'mrr@{REPORT_TOPK}_std'],
            'elapsed_seconds': row['elapsed_seconds_std'],
        },
    }
    (AVG_RESULTS_DIR / f"{row['dataset']}_average.json").write_text(
        json.dumps(dataset_payload, indent=2),
        encoding='utf-8'
    )

    dataset_avg_dir = RESULTS_DIR / row['dataset'] / 'average_results_3_runs'
    dataset_avg_dir.mkdir(parents=True, exist_ok=True)
    dataset_avg_row_df = pd.DataFrame([
        {
            'Dataset': row['dataset'],
            'Model': row['model'],
            'Runs': row['runs'],
            f'Recall@{REPORT_TOPK}_mean': round(row[f'recall@{REPORT_TOPK}_mean'], 6),
            f'Recall@{REPORT_TOPK}_std': round(row[f'recall@{REPORT_TOPK}_std'], 6),
            f'NDCG@{REPORT_TOPK}_mean': round(row[f'ndcg@{REPORT_TOPK}_mean'], 6),
            f'NDCG@{REPORT_TOPK}_std': round(row[f'ndcg@{REPORT_TOPK}_std'], 6),
            f'HR@{REPORT_TOPK}_mean': round(row[f'hr@{REPORT_TOPK}_mean'], 6),
            f'HR@{REPORT_TOPK}_std': round(row[f'hr@{REPORT_TOPK}_std'], 6),
            f'MRR@{REPORT_TOPK}_mean': round(row[f'mrr@{REPORT_TOPK}_mean'], 6),
            f'MRR@{REPORT_TOPK}_std': round(row[f'mrr@{REPORT_TOPK}_std'], 6),
            'elapsed_seconds_mean': round(row['elapsed_seconds_mean'], 6),
            'elapsed_seconds_std': round(row['elapsed_seconds_std'], 6),
        }
    ])
    dataset_avg_row_df.to_csv(dataset_avg_dir / 'average_metrics_3_runs.csv', index=False)
    (dataset_avg_dir / 'average_metrics_3_runs.json').write_text(
        json.dumps(dataset_payload, indent=2),
        encoding='utf-8'
    )

display(summary_df)
display(average_report_df)
print(f'Global 3-run average reports saved to: {AVG_RESULTS_DIR}')
print(f'Per-dataset average folders saved under: {RESULTS_DIR / DATASETS[0] / "average_results_3_runs"} ...')


In [ ]:
def read_repo_split_stats(dataset: str):
    folder = REPO_DIR / 'data' / dataset

    def parse_split(path: Path):
        users = set()
        items = set()
        interactions = 0
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                uid = int(parts[0])
                item_list = [int(x) for x in parts[1:]]
                if not item_list:
                    continue
                users.add(uid)
                items.update(item_list)
                interactions += len(item_list)
        return users, items, interactions

    train_users, train_items, train_interactions = parse_split(folder / 'train.txt')
    test_users, test_items, test_interactions = parse_split(folder / 'test.txt')
    all_users = train_users | test_users
    all_items = train_items | test_items
    total_interactions = train_interactions + test_interactions
    density = total_interactions / (len(all_users) * len(all_items)) if all_users and all_items else 0.0

    return {
        'dataset': dataset,
        'users': len(all_users),
        'items': len(all_items),
        'train_interactions': train_interactions,
        'test_interactions': test_interactions,
        'total_interactions': total_interactions,
        'density': density,
    }


dataset_stats_df = pd.DataFrame(
    [read_repo_split_stats(dataset) for dataset in DATASETS]
).sort_values('dataset').reset_index(drop=True)
dataset_stats_df.to_csv(RESULTS_DIR / 'dataset_characteristics.csv', index=False)

analysis_df = summary_df.merge(dataset_stats_df, on='dataset', how='left')
analysis_df.to_csv(RESULTS_DIR / 'lightgcn_phase3_part1_analysis.csv', index=False)

display(dataset_stats_df)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_specs = [
    (f'recall@{REPORT_TOPK}_mean', f'Recall@{REPORT_TOPK}', '#3366cc'),
    (f'ndcg@{REPORT_TOPK}_mean', f'NDCG@{REPORT_TOPK}', '#dc3912'),
    (f'hr@{REPORT_TOPK}_mean', f'HR@{REPORT_TOPK}', '#009688'),
    (f'mrr@{REPORT_TOPK}_mean', f'MRR@{REPORT_TOPK}', '#8e24aa'),
    ('elapsed_seconds_mean', 'Runtime (s)', '#ff9900'),
    ('density', 'Density', '#109618'),
]

for ax, (col, title, color) in zip(axes.flatten(), plot_specs):
    bars = ax.bar(analysis_df['dataset'], analysis_df[col], color=color)
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for bar, value in zip(bars, analysis_df[col]):
        label = f'{value:.4f}' if value < 1000 else f'{value:,.0f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'analysis_overview.png', dpi=150, bbox_inches='tight')
plt.show()

corr_cols = [
    'users', 'items', 'total_interactions', 'density',
    f'recall@{REPORT_TOPK}_mean', f'ndcg@{REPORT_TOPK}_mean',
    f'hr@{REPORT_TOPK}_mean', f'mrr@{REPORT_TOPK}_mean'
]
print('Correlation summary:')
display(analysis_df[corr_cols].corr(numeric_only=True))


In [ ]:
for history_path in sorted(RESULTS_DIR.glob('*/*/history.csv')):
    history_df = pd.read_csv(history_path)
    if history_df.empty:
        continue
    dataset = history_path.parent.parent.name
    run_name = history_path.parent.name
    plt.figure(figsize=(7, 4))
    plt.plot(history_df['eval_epoch'], history_df['recall@20'], label='recall@20')
    plt.plot(history_df['eval_epoch'], history_df['ndcg@20'], label='ndcg@20')
    plt.xlabel('Evaluation Epoch')
    plt.title(f'{dataset} - {run_name}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{dataset}_{run_name}_convergence.png', dpi=150, bbox_inches='tight')
    plt.close()

run_metadata = {
    'repo_url': REPO_URL,
    'datasets': DATASETS,
    'fixed_seed': FIXED_SEED,
    'run_ids': RUN_IDS,
    'topks': TOPKS,
    'report_topk': REPORT_TOPK,
    'early_stopping': EARLY_STOPPING,
    'torch_version': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'lastfm_input_dir': str(LASTFM_PHASE2_INPUT_DIR),
    'average_results_dir': str(AVG_RESULTS_DIR),
}
(RESULTS_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

archive_base = ROOT / 'phase3_part1_lightgcn_results'
archive_zip = archive_base.with_suffix('.zip')
if archive_zip.exists():
    archive_zip.unlink()
shutil.make_archive(str(archive_base), 'zip', RESULTS_DIR)
print(f'Results archive: {archive_zip}')
print('Finished Phase 3 Part 1.')
